# Experimentos de mi proyecto (M5, CA_1) registrados en MLflow

Una corrida por cada modelo de la Semana 10: linea base, modelo lineal, ensamblaje y red neuronal. Cada corrida registra hiperparametros, como se dividieron los datos, la huella del archivo, el MAE en entrenamiento y prueba, los segundos y el pipeline entrenado. Al final se marca la corrida final y se comprueba que reproduce su metrica.

Requiere el servidor de MLflow (variable `MLFLOW_TRACKING_URI`).

In [1]:
import hashlib
import logging
import time

import mlflow
import mlflow.sklearn
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

from pipeline_proyecto import (DIAS_DE_PRUEBA, RUTA_DATOS, LineaBaseMedia28,
                               cargar_y_dividir, construir_preparacion)

logging.getLogger("mlflow").setLevel(logging.ERROR)

MUESTRA_ENTRENAMIENTO = 200_000
X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir(maximo_entrenamiento=MUESTRA_ENTRENAMIENTO)
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)


def huella(ruta):
    md5 = hashlib.md5()
    with open(ruta, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(1 << 20), b""):
            md5.update(bloque)
    return md5.hexdigest()[:12]


huella_datos = huella(RUTA_DATOS)
print("Huella de", RUTA_DATOS, ":", huella_datos)
EXPERIMENTO = "m5-demanda-CA_1"
mlflow.set_experiment(EXPERIMENTO)
print("Registro de MLflow:", mlflow.get_tracking_uri())

Entrenamiento: (200000, 9) Prueba: (170744, 9)


Huella de ../data/processed/obt_sales_CA_1.csv.gz : 2aebb531e268


Registro de MLflow: http://127.0.0.1:5051


In [2]:
def evaluar(nombre, modelo, preparar=True):
    pasos = [("preparar", construir_preparacion())] if preparar else []
    pipeline = Pipeline(pasos + [("modelo", modelo)])
    with mlflow.start_run(run_name=nombre):
        mlflow.log_params({
            **modelo.get_params(),
            "division": "por fecha",
            "dias_de_prueba": DIAS_DE_PRUEBA,
            "muestra_entrenamiento": MUESTRA_ENTRENAMIENTO,
            "huella_datos": huella_datos,
            "filas_entrenamiento": len(X_ent),
            "filas_prueba": len(X_prueba),
        })
        inicio = time.perf_counter()
        pipeline.fit(X_ent, y_ent)
        segundos = time.perf_counter() - inicio
        mae_ent = mean_absolute_error(y_ent, pipeline.predict(X_ent))
        mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))
        mlflow.log_metrics({"mae_entrenamiento": mae_ent, "mae_prueba": mae_prueba, "segundos": segundos})
        mlflow.sklearn.log_model(pipeline, "modelo")
    print(f"{nombre:20s} entrenamiento {mae_ent:.4f}   prueba {mae_prueba:.4f}   {segundos:6.2f} s")

In [3]:
evaluar("linea base", LineaBaseMedia28(), preparar=False)
evaluar("ridge", Ridge(alpha=1.0))
evaluar("gradient boosting", GradientBoostingRegressor(random_state=42))
evaluar("red neuronal", MLPRegressor(hidden_layer_sizes=(64,), max_iter=1000, random_state=42))

linea base           entrenamiento 1.1793   prueba 1.1082     0.00 s


ridge                entrenamiento 1.1921   prueba 1.1190     0.20 s


gradient boosting    entrenamiento 1.1470   prueba 1.0895    16.63 s


red neuronal         entrenamiento 1.1307   prueba 1.0713   112.06 s


## Elegir la corrida final desde el registro

La red neuronal tiene el menor error en prueba, pero solo por 0.018 unidades (1.7%) sobre el gradient boosting. Elijo el **gradient boosting** como final: casi el mismo error, 8 veces menos tiempo de entrenamiento y es explicable (importancias de las columnas). La decision y su evidencia quedan en el registro con la etiqueta `candidato` = `final`.

In [4]:
experimento = mlflow.get_experiment_by_name(EXPERIMENTO)
corridas = mlflow.search_runs(experiment_ids=[experimento.experiment_id], order_by=["metrics.mae_prueba ASC"])
corridas[["run_id", "tags.mlflow.runName", "metrics.mae_entrenamiento", "metrics.mae_prueba", "metrics.segundos"]]

,run_id,tags.mlflow.runName,metrics.mae_entrenamiento,metrics.mae_prueba,metrics.segundos
0,8bf3c2862b15404ab177daeb0b6bf7e9,red neuronal,1.130661,1.071292,112.055090
1,7ca7b0e729c148d186cf4bd02fbce8ec,gradient boosting,1.146963,1.089455,16.628563
2,5c95e0730dbf4bc082f22b234a3b0109,linea base,1.179316,1.108168,0.000466
3,ed0d61d5527041c2bf70affc1de442eb,ridge,1.192135,1.119022,0.200076


In [5]:
mejor_run_id = corridas.loc[corridas["tags.mlflow.runName"] == "gradient boosting", "run_id"].iloc[0]
mlflow.MlflowClient().set_tag(mejor_run_id, "candidato", "final")
modelo_final = mlflow.sklearn.load_model("runs:/" + mejor_run_id + "/modelo")
mae_reproducido = mean_absolute_error(y_prueba, modelo_final.predict(X_prueba))
registrado = corridas.loc[corridas["run_id"] == mejor_run_id, "metrics.mae_prueba"].iloc[0]
print("Corrida final: gradient boosting", mejor_run_id)
print("MAE registrado:", registrado, "| reproducido:", mae_reproducido)
assert abs(mae_reproducido - registrado) < 1e-9

Corrida final: gradient boosting 7ca7b0e729c148d186cf4bd02fbce8ec
MAE registrado: 1.0894550357354396 | reproducido: 1.0894550357354396


In [6]:
corridas = mlflow.search_runs(experiment_ids=[experimento.experiment_id], order_by=["metrics.mae_prueba ASC"])
corridas.to_csv("corridas_proyecto.csv", index=False)
print(len(corridas), "corridas exportadas a corridas_proyecto.csv")

4 corridas exportadas a corridas_proyecto.csv
